# V3 Stage 2: Full Value Logit Lens

Thin notebook — all logic lives in `stage2_logit_lens.py`.
Edit CONFIG, run all cells.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════════════════

CONFIG = {
    "model": "Qwen/Qwen3-4B",
    "points": [(5, 7), (5, 10), (7, 10)],
    "trials": 50,
    "gpu": 0,        # Physical GPU index (None = auto-detect)
    "n_ctx": 8192,
}

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SETUP
# ═══════════════════════════════════════════════════════════════════════════

import sys, os
from pathlib import Path

notebook_dir = Path(os.getcwd()).resolve()
if notebook_dir.name == "v3":
    PROJECT_ROOT = notebook_dir.parent
else:
    for p in [notebook_dir, notebook_dir.parent, notebook_dir.parent.parent]:
        if (p / "mechanistic_probing_v2" / "core").exists():
            PROJECT_ROOT = p
            break
    else:
        raise RuntimeError("Cannot find project root.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT / "v3") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "v3"))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LOAD MODEL
# ═══════════════════════════════════════════════════════════════════════════

from mechanistic_probing_v2.core.model_loader import load_model

model, tokenizer, info = load_model(
    CONFIG["model"],
    n_ctx=CONFIG.get("n_ctx", 8192),
    gpu_idx=CONFIG.get("gpu"),
)
print(f"\nReady: {info.n_layers} layers, {info.n_heads} heads, d_model={info.d_model}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RUN STAGE 2
# ═══════════════════════════════════════════════════════════════════════════

from stage2_logit_lens import run_stage2

results = run_stage2(CONFIG, model=model, tokenizer=tokenizer, info=info)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# QUICK ANALYSIS: Last-layer P(v_i) for PI failures
# ═══════════════════════════════════════════════════════════════════════════

import numpy as np

for pt in CONFIG["points"]:
    k, u = pt
    pi_failures = [r for r in results["analyses"]
                   if r["condition"] == "PI"
                   and r["num_keys"] == k and r["num_updates"] == u
                   and not r["correct"]
                   and r["total_value_prob"] > 0.05]  # filter garbage

    if not pi_failures:
        print(f"\n{k}k_{u}u: no usable PI failures")
        continue

    # Average P(v_i) at last layer across PI failures
    all_probs = []
    for r in pi_failures:
        vp = r["value_probs_by_layer"]
        all_probs.append([vp[vi][-1] for vi in range(len(vp))])
    avg = np.mean(all_probs, axis=0)

    print(f"\n=== {k}k_{u}u PI failures ({len(pi_failures)} trials, garbage filtered) ===")
    print(f"Last-layer avg P(v_i):")
    for i, p in enumerate(avg):
        bar = "#" * int(p * 100)
        tag = " <- FIRST" if i == 0 else (" <- LAST (correct)" if i == len(avg) - 1 else "")
        print(f"  v{i:>2}: {p:.4f}  {bar}{tag}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# LAYER TRAJECTORY: P(v_last) vs P(v_competitor) across layers
# ═══════════════════════════════════════════════════════════════════════════

for pt in CONFIG["points"]:
    k, u = pt
    pi_failures = [r for r in results["analyses"]
                   if r["condition"] == "PI"
                   and r["num_keys"] == k and r["num_updates"] == u
                   and not r["correct"]
                   and r["total_value_prob"] > 0.05]

    ri_correct = [r for r in results["analyses"]
                  if r["condition"] == "RI"
                  and r["num_keys"] == k and r["num_updates"] == u
                  and r["correct"]]

    if not pi_failures:
        continue

    n_layers = pi_failures[0]["n_layers"]
    n_values = len(pi_failures[0]["all_values"])

    # Average across trials: shape [n_values, n_layers]
    pi_avg = np.mean([r["value_probs_by_layer"] for r in pi_failures], axis=0)

    # Find dominant competitor at last layer (excluding correct answer)
    last_probs = pi_avg[:, -1].copy()
    correct_idx = n_values - 1
    last_probs_no_correct = last_probs.copy()
    last_probs_no_correct[correct_idx] = -1
    competitor_idx = np.argmax(last_probs_no_correct)

    print(f"\n=== {k}k_{u}u Layer Trajectory (PI failures, {len(pi_failures)} trials) ===")
    print(f"  Correct: v{correct_idx} (last value)")
    print(f"  Competitor: v{competitor_idx} (dominant wrong value)")
    print(f"  v0: first value")
    print()
    print(f"  {'Layer':>5}  {'P(v_last)':>10}  {'P(v_comp)':>10}  {'P(v_0)':>10}  {'Winner':>8}")
    print(f"  {'-'*50}")
    for L in range(0, n_layers, max(1, n_layers // 12)):
        p_last = pi_avg[correct_idx, L]
        p_comp = pi_avg[competitor_idx, L]
        p_first = pi_avg[0, L]
        winner = "v_last" if p_last > p_comp else f"v{competitor_idx}"
        print(f"  L{L:>3}:  {p_last:>10.4f}  {p_comp:>10.4f}  {p_first:>10.4f}  {winner:>8}")
    # Final layer
    L = n_layers - 1
    p_last = pi_avg[correct_idx, L]
    p_comp = pi_avg[competitor_idx, L]
    p_first = pi_avg[0, L]
    winner = "v_last" if p_last > p_comp else f"v{competitor_idx}"
    print(f"  L{L:>3}:  {p_last:>10.4f}  {p_comp:>10.4f}  {p_first:>10.4f}  {winner:>8}  <- FINAL")

    # RI comparison
    if ri_correct:
        ri_avg = np.mean([r["value_probs_by_layer"] for r in ri_correct], axis=0)
        print(f"\n  RI correct ({len(ri_correct)} trials): P(v_0) at final layer = {ri_avg[0, -1]:.4f}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CROSSOVER ANALYSIS: at which layer does v_last lose to competitor?
# ═══════════════════════════════════════════════════════════════════════════

for pt in CONFIG["points"]:
    k, u = pt
    pi_failures = [r for r in results["analyses"]
                   if r["condition"] == "PI"
                   and r["num_keys"] == k and r["num_updates"] == u
                   and not r["correct"]
                   and r["total_value_prob"] > 0.05]

    if not pi_failures:
        continue

    n_layers = pi_failures[0]["n_layers"]
    n_values = len(pi_failures[0]["all_values"])
    correct_idx = n_values - 1

    # Per-trial: find the layer where v_last was leading then lost
    crossover_layers = []
    never_led = 0
    always_led = 0

    for r in pi_failures:
        vp = np.array(r["value_probs_by_layer"])
        p_last = vp[correct_idx, :]  # P(v_last) at each layer
        # Max of all other values at each layer
        other = np.delete(vp, correct_idx, axis=0)
        p_best_other = other.max(axis=0)

        leading = p_last > p_best_other
        if not leading.any():
            never_led += 1
            continue
        if leading[-1]:
            always_led += 1
            continue
        # Find last layer where v_last was leading
        last_leading = np.where(leading)[0][-1]
        crossover_layers.append(last_leading)

    n = len(pi_failures)
    print(f"\n=== {k}k_{u}u Crossover Analysis ({n} PI failures) ===")
    print(f"  Never led: {never_led}/{n} — v_last was never the top value at any layer")
    print(f"  Overtaken: {len(crossover_layers)}/{n} — v_last led then lost")
    print(f"  Always led: {always_led}/{n} — v_last led through final layer (correct)")

    if crossover_layers:
        print(f"  Crossover layer: mean={np.mean(crossover_layers):.1f}, "
              f"median={np.median(crossover_layers):.0f}, "
              f"range=[{min(crossover_layers)}, {max(crossover_layers)}]")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# GARBAGE THRESHOLD: total_value_prob vs predicted position
# ═══════════════════════════════════════════════════════════════════════════

all_pi = [r for r in results["analyses"] if r["condition"] == "PI" and not r["correct"]]

print(f"All PI failures: {len(all_pi)}")
print(f"\n{'total_value_prob':>18}  {'count':>5}  {'has_pred_idx':>12}  {'avg_relpos':>10}")
print("-" * 55)

bins = [(0, 0.01), (0.01, 0.05), (0.05, 0.10), (0.10, 0.30), (0.30, 0.60), (0.60, 1.0)]
for lo, hi in bins:
    subset = [r for r in all_pi if lo <= r["total_value_prob"] < hi]
    if not subset:
        continue
    has_idx = sum(1 for r in subset if r["predicted_idx"] is not None)
    positions = [r["predicted_relative_pos"] for r in subset if r["predicted_relative_pos"] is not None]
    avg_pos = f"{np.mean(positions):.2f}" if positions else "n/a"
    print(f"  [{lo:.2f}, {hi:.2f})  {len(subset):>5}  {has_idx:>12}  {avg_pos:>10}")